In [ ]:
"""
DEMO to evaluate transfered EQCCT PT models from their original TF counterparts 
"""
# 1) ENVIRONMENT & VERSIONS
# -------------------------
import sys, torch, tensorflow as tf
print("Python:", sys.version)
print("Torch: ", torch.__version__)
print("TF:    ", tf.__version__)

import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# lock seeds for full reproducibility
SEED = 42
dataset_choice = 0 
n_examples = 10
SAMPLE_RATE = 100
ERROR_THRESHOLDS = [1.0, 5.0, 10.0, 15.0]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)


# 2) DATASET + TRACE LIST
# ------------------------
import seisbench.data as sbd

# choose STEAD or TXED here
if dataset_choice == 1: 
    dataset = sbd.STEAD(sampling_rate=SAMPLE_RATE, component_order="ZNE")
    ds_name = 'STEAD'
else: 
    dataset = sbd.TXED(sampling_rate=SAMPLE_RATE, component_order="ZNE")
    ds_name = 'TXED'

# Get waveform subset that has both P & S picks 
md = dataset.metadata
good_p = md["trace_p_arrival_sample"].notna() & (md["trace_p_arrival_sample"] > 0)
good_s = md["trace_s_arrival_sample"].notna() & (md["trace_s_arrival_sample"] > 0)
valid = md[good_p & good_s]["trace_name"].tolist()
random.shuffle(valid)


print(f"\n{len(valid)} valid {ds_name} 3-C traces (seed={SEED})")


def batch_iter(trace_names, batch_size=32):
    def norm_std(x):
        x = x - x.mean(axis=0, keepdims=True)
        return (x / (x.std(axis=0, keepdims=True) + 1e-8)).astype("float32")

    for i in range(0, len(trace_names), batch_size):
        chunk = trace_names[i:i+batch_size]
        idxs  = [dataset.get_idx_from_trace_name(t) for t in chunk]
        wfs   = dataset.get_waveforms(idxs)  # list of 3-component arrays
        wfs   = np.stack(wfs).reshape(len(idxs),3,-1).transpose(0,2,1)
        yield chunk, norm_std(wfs), dataset.metadata.loc[idxs]


# 3) LOAD MODELS
# --------------
from predictor_pt import EQCCTModelP
from smodel import EQCCTModelS
from predictor_tf import load_eqcct_model 

TF_P_H5 = "ModelPS/test_trainer_024.h5"
TF_S_H5 = "ModelPS/test_trainer_021.h5"
PT_P_PTH = "ModelPS/modelP_fixed_bn.pth"
PT_S_PTH = "new_smodel.pth"

tf_p, tf_s = load_eqcct_model(TF_P_H5, TF_S_H5)
pt_p = EQCCTModelP().eval(); pt_p.load_state_dict(torch.load(PT_P_PTH))
pt_s = EQCCTModelS().eval(); pt_s.load_state_dict(torch.load(PT_S_PTH))

device = "cuda" if torch.cuda.is_available() else "cpu"
pt_p, pt_s = pt_p.to(device), pt_s.to(device)


# 4) INFERENCE & PICKING
# -----------------------
@torch.no_grad()
def run_pt(model, wf):
    return model(torch.from_numpy(wf).to(device)).cpu().numpy().squeeze(-1)

def run_tf(model, wf):
    # wf: (B,6000,3) numpy
    return model(wf, training=False).numpy().squeeze(-1)

def pick_from_prob(prob, thresh=0.4):
    # returns sample index of max confidence above thresh
    masked = np.where(prob>=thresh, prob, 0)
    idx    = masked.argmax(axis=1)
    return idx

# 5) ERROR COMPUTATION
# --------------------
all_errors = {"TF-P":[], "PT-P":[], "TF-S":[], "PT-S":[]} # Corrected dictionary initialization
print(f"Evaluating TF & PT models on {ds_name} with sampling rate of {SAMPLE_RATE}Hz…")
for names, wfs, meta in tqdm(batch_iter(valid), total=len(valid)//32):
    # ground truth
    p_true = meta["trace_p_arrival_sample"].values.astype(int)
    s_true = meta["trace_s_arrival_sample"].values.astype(int)

    # Run P-wave models
    tf_p_probs = run_tf(tf_p, wfs)
    pt_p_probs = run_pt(pt_p, wfs)
    tf_p_idx = pick_from_prob(tf_p_probs, 0.4)
    pt_p_idx = pick_from_prob(pt_p_probs, 0.4)

    errs_tf_p = (p_true - tf_p_idx)/SAMPLE_RATE
    errs_pt_p = (p_true - pt_p_idx)/SAMPLE_RATE
    all_errors["TF-P"].extend(errs_tf_p.tolist())
    all_errors["PT-P"].extend(errs_pt_p.tolist())

    # run S-wave models
    tf_s_probs = run_tf(tf_s, wfs)
    pt_s_probs = run_pt(pt_s, wfs)
    tf_s_idx = pick_from_prob(tf_s_probs, 0.4)
    pt_s_idx = pick_from_prob(pt_s_probs, 0.4)

    errs_tf_s = (s_true - tf_s_idx)/SAMPLE_RATE
    errs_pt_s = (s_true - pt_s_idx)/SAMPLE_RATE
    all_errors["TF-S"].extend(errs_tf_s.tolist())
    all_errors["PT-S"].extend(errs_pt_s.tolist())

# 6) SUMMARY STATISTICS
# ---------------------
for ERROR_THRESHOLD in ERROR_THRESHOLDS: 
    print(f"\nEvaluating with ERROR_THRESHOLD = {ERROR_THRESHOLD}")
    filtered_all_errors = {}
    for name, errs in all_errors.items():
        errs = np.array(errs)
        
        # Filter out the MAE's greater than +/- 5s 
        filtered_errs = [error for error in errs if abs(error) <= ERROR_THRESHOLD]
        filtered_errs = np.array(filtered_errs)

        # Store the filtered errors in a new dict
        filtered_all_errors[name] = np.array(filtered_errs)

        mae  = np.mean(np.abs(filtered_all_errors[name]))
        std  = np.std(filtered_all_errors[name])
        print(f"{name:4s}: MAE={mae:.3f}s,  σ={std:.3f}s")
        print(f"*With filtering, MAE was calculated using {len(filtered_errs)}/{len(errs)} samples")

    # 7) PLOT HISTOGRAMS + EXAMPLES
    # ------------------------------
    fig, (ax_p, ax_s) = plt.subplots(1,2, figsize=(16, 5))

    # P-wave histograms
    tf_p_label = f'TF-P ({len(filtered_all_errors["TF-P"])}/{len(all_errors["TF-P"])})'
    pt_p_label = f'PT-P ({len(filtered_all_errors["PT-P"])}/{len(all_errors["PT-P"])})'
    ax_p.hist(filtered_all_errors["TF-P"], bins=50, alpha=0.6, label=tf_p_label)
    ax_p.hist(filtered_all_errors["PT-P"], bins=50, alpha=0.6, label=pt_p_label)
    ax_p.set_xlabel("P-pick error (s)")
    ax_p.set_ylabel("count")
    ax_p.set_title(f"P-Picking Error on {ds_name} (abs error <= {ERROR_THRESHOLD}s)")
    ax_p.legend()

    # S-wave histograms
    tf_s_label = f'TF-S ({len(filtered_all_errors["TF-S"])}/{len(all_errors["TF-S"])})'
    pt_s_label = f'PT-S ({len(filtered_all_errors["PT-S"])}/{len(all_errors["PT-S"])})'
    ax_s.hist(filtered_all_errors["TF-S"], bins=50, alpha=0.6, label=tf_s_label)
    ax_s.hist(filtered_all_errors["PT-S"], bins=50, alpha=0.6, label=pt_s_label)
    ax_s.set_xlabel("S-pick error (s)")
    ax_s.set_ylabel("count")
    ax_s.set_title(f"S-Picking Error on {ds_name} (abs error <= {ERROR_THRESHOLD}s)")
    ax_s.legend()
    plt.savefig(f"{ERROR_THRESHOLD}s_hist.png", dpi=300)
    plt.show()

    # 8) PLOT P-WAVE EXAMPLES
    print(f"\nPlotting P-wave examples for threshold {ERROR_THRESHOLD}s...")
    # grab their waveforms & picks again
    plotting_p_examples = []
    wfs_p_ex, p_true_p_ex, tf_p_idx_ex, pt_p_idx_ex = [], [], [], []

    for name in tqdm(valid, desc='Filtering P examples'):
        if len(plotting_p_examples) >= n_examples: 
            break 

        idx = dataset.get_idx_from_trace_name(name)
        wf = dataset.get_waveforms([idx])[0]            # (3,6000)
        wf = wf.T                                       # (6000,3)
        wf = (wf - wf.mean(0)) / (wf.std(0) + 1e-8)     # normalize
        wf = wf.astype(np.float32)

        # Get ground truth
        p_true = int(dataset.metadata.loc[idx,"trace_p_arrival_sample"])

        # run inference
        prob_tf = run_tf(tf_p, wf[np.newaxis,...])
        prob_pt = run_pt(pt_p, wf[np.newaxis,...])
        tf_idx  = pick_from_prob(prob_tf, 0.4)[0]
        pt_idx  = pick_from_prob(prob_pt, 0.4)[0]

        # Compute errors 
        err_tf = (p_true - tf_idx) / SAMPLE_RATE
        err_pt = (p_true - pt_idx) / SAMPLE_RATE

        # Check if both errors are within the acceptable error range 
        if abs(err_tf) <= ERROR_THRESHOLD and abs(err_pt) <= ERROR_THRESHOLD: 
            wfs_p_ex.append(wf) 
            p_true_p_ex.append(p_true)
            tf_p_idx_ex.append(tf_idx)
            pt_p_idx_ex.append(pt_idx)
            plotting_p_examples.append(name)

    fig, axs = plt.subplots(n_examples, 1, figsize=(10, 2*n_examples), sharex=True)
    for i, ax in enumerate(axs):
        for comp in range(3):
            ax.plot(wfs_p_ex[i][:,comp] + comp*2.5, color='k', lw=0.5)
        ax.axvline(p_true_p_ex[i],    color='red', linestyle='-', label='GT P-pick')
        ax.axvline(tf_p_idx_ex[i],    color='blue', linestyle='--',  label='TF-P')
        ax.axvline(pt_p_idx_ex[i],    color='green', linestyle=':',  label='PT-P')
        ax.set_ylabel(f"Trace {i+1}")
        ax.legend(loc='upper right', fontsize='small')

        if i == 0: 
            ax.set_title(f"{n_examples} {ds_name} P Examples with TF & PT Picks (abs error <= {ERROR_THRESHOLD}s)")

        err_tf = (p_true_p_ex[i] - tf_p_idx_ex[i]) / SAMPLE_RATE
        err_pt = (p_true_p_ex[i] - pt_p_idx_ex[i]) / SAMPLE_RATE

        ax.text(
            0.01, 0.95,
            f"TF err: {err_tf:.3f}s\nPT err: {err_pt:.3f}s",
            transform=ax.transAxes,
            va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
        ax.set_ylabel(f"Trace {i+1}", rotation=0, labelpad=30, va='center')

    axs[-1].set_xlabel("Sample Index")
    plt.savefig(f"{ERROR_THRESHOLD}s_p_examples.png", dpi=300)
    plt.show()

    # 9) PLOT S-WAVE EXAMPLES
    # ------------------------------
    print(f"\nPlotting S-wave examples for threshold {ERROR_THRESHOLD}s...")
    plotting_s_examples = []
    wfs_s_ex, p_true_s_ex, tf_s_idx_ex, pt_s_idx_ex = [], [], [], []

    for name in tqdm(valid, desc='Filtering S examples'):
        if len(plotting_s_examples) >= n_examples: 
            break 

        idx = dataset.get_idx_from_trace_name(name)
        wf = dataset.get_waveforms([idx])[0]
        wf = wf.T
        wf = (wf - wf.mean(0)) / (wf.std(0) + 1e-8)
        wf = wf.astype(np.float32)

        s_true = int(dataset.metadata.loc[idx,"trace_s_arrival_sample"])

        prob_tf = run_tf(tf_s, wf[np.newaxis,...])
        prob_pt = run_pt(pt_s, wf[np.newaxis,...])
        tf_idx  = pick_from_prob(prob_tf, 0.4)[0]
        pt_idx  = pick_from_prob(prob_pt, 0.4)[0]

        err_tf = (s_true - tf_idx) / SAMPLE_RATE
        err_pt = (s_true - pt_idx) / SAMPLE_RATE

        if abs(err_tf) <= ERROR_THRESHOLD and abs(err_pt) <= ERROR_THRESHOLD: 
            wfs_s_ex.append(wf) 
            p_true_s_ex.append(s_true)
            tf_s_idx_ex.append(tf_idx)
            pt_s_idx_ex.append(pt_idx)
            plotting_s_examples.append(name)

    fig, axs = plt.subplots(n_examples, 1, figsize=(10, 2*n_examples), sharex=True)
    for i, ax in enumerate(axs):
        for comp in range(3):
            ax.plot(wfs_s_ex[i][:,comp] + comp*2.5, color='k', lw=0.5)
        ax.axvline(p_true_s_ex[i],    color='red', linestyle='-', label='GT S-pick')
        ax.axvline(tf_s_idx_ex[i],    color='blue', linestyle='--',  label='TF-S')
        ax.axvline(pt_s_idx_ex[i],    color='green', linestyle=':',  label='PT-S')
        ax.set_ylabel(f"Trace {i+1}")
        ax.legend(loc='upper right', fontsize='small')

        if i == 0: 
            ax.set_title(f"{n_examples} {ds_name} S Examples with TF & PT Picks (abs error <= {ERROR_THRESHOLD}s)")

        err_tf = (p_true_s_ex[i] - tf_s_idx_ex[i]) / SAMPLE_RATE
        err_pt = (p_true_s_ex[i] - pt_s_idx_ex[i]) / SAMPLE_RATE

        ax.text(
            0.01, 0.95,
            f"TF err: {err_tf:.3f}s\nPT err: {err_pt:.3f}s",
            transform=ax.transAxes,
            va='top', ha='left',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=2)
        )
        ax.set_ylabel(f"Trace {i+1}", rotation=0, labelpad=30, va='center')

    axs[-1].set_xlabel("Sample Index")
    plt.tight_layout()
    plt.savefig(f"{ERROR_THRESHOLD}s_s_examples.png", dpi=300)
    plt.show()

Python: 3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]
Torch:  2.7.1+cu126
TF:     2.19.0

312231 valid TXED 3-C traces (seed=42)
Evaluating TF & PT models on TXED with sampling rate of 100Hz…


 38%|███▊      | 3705/9757 [07:23<12:04,  8.36it/s]


KeyboardInterrupt: 

: 